In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os
from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.features.features_v1 import *
from src.features.features_v2 import *
from src.pipeline.calculate_evs import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 14 teams with confirmed lineups


### Load Model

### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if us_file is None:
    raise FileNotFoundError(f"No NBA_US file found for {today}")
if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
usData = pd.read_csv(us_file)
dfsData = pd.read_csv(dfs_file)

print(f"Loaded: {us_file.name}")
print(f"Loaded: {dfs_file.name}")
dfsData.head()

Loaded: NBA_US_20251206_072956.csv
Loaded: NBA_DFS_20251206_073400.csv


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Michael Porter Jr,Over,25.5,-137,2025-12-06,2025-12-06T15:28:29Z,2025-12-06 07:34:00
1,PrizePicks,player_points,Michael Porter Jr,Under,25.5,-137,2025-12-06,2025-12-06T15:28:29Z,2025-12-06 07:34:00
2,PrizePicks,player_points,Trey Murphy III,Over,20.5,-137,2025-12-06,2025-12-06T15:28:29Z,2025-12-06 07:34:00
3,PrizePicks,player_points,Trey Murphy III,Under,20.5,-137,2025-12-06,2025-12-06T15:28:29Z,2025-12-06 07:34:00
4,PrizePicks,player_points,Saddiq Bey,Over,17.5,-137,2025-12-06,2025-12-06T15:28:29Z,2025-12-06 07:34:00


In [4]:
from src.features.feature_engine import FeatureEngine

engine = FeatureEngine({
    "min_model": "src/models/saved/min_model.pkl",
    "usg_model": "src/models/saved/usg_model.pkl",
    "fga_model": "src/models/saved/fga_model.pkl",
    "ngboost_model_wrapper": "src/models/saved/pts_model_wrapper.pkl"
})

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Top EVs for 2 leg bets

### Underdog picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

# underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv('data/props/ev_analysis/underdogPairs.csv', index=False)
underdogPairs

Computing predictions for 78 players...
[MIN] No data found for Egor Demin
Found 77 valid players
Generated 2530 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,SIDE 1,SIDE 2,PREDICTION 1,PREDICTION 2,EDGE 1,EDGE 2,ODDS 1,ODDS 2,IMPLIED_PROB 1,IMPLIED_PROB 2,TOTAL_EDGE
1311,Marvin Bagley III,Tyler Herro,13.5,23.5,under,over,7.07,30.00,6.43,6.50,-113,-110,0.52,0.51,12.93
358,Danny Wolf,Kevin Porter Jr.,11.5,20.5,under,under,5.50,14.42,6.00,6.08,-112,-104,0.52,0.50,12.08
565,Noah Clowney,Jaden McDaniels,16.5,13.5,under,over,10.74,19.39,5.76,5.89,-110,110,0.51,0.46,11.65
2183,Julius Randle,Maxime Raynaud,20.5,10.5,over,under,26.11,5.19,5.61,5.31,-110,-105,0.51,0.50,10.92
1994,Gary Harris,Aaron Holiday,5.5,9.5,under,under,0.49,4.21,5.01,5.29,-130,-110,0.55,0.51,10.30


### Prizepicks picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)


prizepicksPairs.to_csv('data/props/ev_analysis/prizepicksPairs.csv', index=False)
prizepicksPairs

Computing predictions for 98 players...
[MIN] No data found for Egor Demin
Found 97 valid players
Generated 4028 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,SIDE 1,SIDE 2,PREDICTION 1,PREDICTION 2,EDGE 1,EDGE 2,ODDS 1,ODDS 2,IMPLIED_PROB 1,IMPLIED_PROB 2,TOTAL_EDGE
3633,Jaden McDaniels,Jaime Jaquez Jr.,12.5,13.5,over,over,19.39,22.29,6.89,8.79,-120,-114,0.53,0.52,15.68
1547,Marvin Bagley III,Tyler Herro,13.5,23.5,under,over,7.07,30.00,6.43,6.50,-113,-110,0.52,0.51,12.93
608,Danny Wolf,Kevin Porter Jr.,11.5,20.5,under,under,5.50,14.42,6.00,6.08,-112,-104,0.52,0.50,12.08
374,Noah Clowney,Julius Randle,16.5,20.5,under,over,10.74,26.11,5.76,5.61,-110,-110,0.51,0.51,11.37
3970,Maxime Raynaud,Aaron Holiday,10.5,9.5,under,under,5.19,4.21,5.31,5.29,-105,-110,0.50,0.51,10.60
1297,Nickeil Alexander-Walker,Ryan Nembhard,20.5,9.5,over,under,25.30,4.54,4.80,4.96,-118,-137,0.53,0.56,9.76
1453,Kyshawn George,Jalen Duren,15.5,18.5,under,under,10.77,13.90,4.73,4.60,-118,100,0.53,0.49,9.33
111,Trey Murphy III,Ryan Rollins,20.5,20.5,over,under,25.03,15.91,4.53,4.59,-135,-105,0.56,0.50,9.12
181,Saddiq Bey,Jimmy Butler III,17.5,20.5,under,under,13.26,16.22,4.24,4.28,-117,110,0.53,0.46,8.52
3888,Andrew Wiggins,Kevin Durant,16.5,25.5,over,under,20.63,21.32,4.13,4.18,-104,-110,0.50,0.51,8.31


## 3 leg parlay

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') ]

underdogTrios = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogTrios.to_csv('data/props/ev_analysis/underdogTrios.csv', index=False)
underdogTrios.head()

Computing predictions for 78 players...
[MIN] No data found for Egor Demin
Found 77 valid players
Generated 45986 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,SIDE 1,SIDE 2,SIDE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,EDGE 1,EDGE 2,EDGE 3,ODDS 1,ODDS 2,ODDS 3,IMPLIED_PROB 1,IMPLIED_PROB 2,IMPLIED_PROB 3,TOTAL_EDGE
32552,Marvin Bagley III,Kevin Porter Jr.,Tyler Herro,13.5,20.5,23.5,under,under,over,7.07,14.42,30.00,6.43,6.08,6.50,-113,-104,-110,0.52,0.50,0.51,19.01
10555,Danny Wolf,Jaden McDaniels,Tyler Herro,11.5,13.5,23.5,under,over,over,5.50,19.39,30.00,6.00,5.89,6.50,-112,110,-110,0.52,0.46,0.51,18.39
15879,Noah Clowney,Julius Randle,Tyler Herro,16.5,20.5,23.5,under,over,over,10.74,26.11,30.00,5.76,5.61,6.50,-110,-110,-110,0.51,0.51,0.51,17.87
42980,Gary Harris,Tyler Herro,Aaron Holiday,5.5,23.5,9.5,under,over,under,0.49,30.00,4.21,5.01,6.50,5.29,-130,-110,-110,0.55,0.51,0.51,16.80
26593,Nickeil Alexander-Walker,Ryan Rollins,Tyler Herro,20.5,20.5,23.5,over,under,over,25.30,15.91,30.00,4.80,4.59,6.50,-118,-105,-110,0.53,0.50,0.51,15.89


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

triosPrizepicks.to_csv('data/props/ev_analysis/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Computing predictions for 98 players...
[MIN] No data found for Egor Demin
Found 97 valid players
Generated 92824 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,SIDE 1,SIDE 2,SIDE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,EDGE 1,EDGE 2,EDGE 3,ODDS 1,ODDS 2,ODDS 3,IMPLIED_PROB 1,IMPLIED_PROB 2,IMPLIED_PROB 3,TOTAL_EDGE
51129,Marvin Bagley III,Jaden McDaniels,Jaime Jaquez Jr.,13.5,12.5,13.5,under,over,over,7.07,19.39,22.29,6.43,6.89,8.79,-113,-120,-114,0.52,0.53,0.52,22.11
21779,Danny Wolf,Kevin Porter Jr.,Jaime Jaquez Jr.,11.5,20.5,13.5,under,under,over,5.50,14.42,22.29,6.00,6.08,8.79,-112,-104,-114,0.52,0.50,0.52,20.87
13785,Noah Clowney,Julius Randle,Jaime Jaquez Jr.,16.5,20.5,13.5,under,over,over,10.74,26.11,22.29,5.76,5.61,8.79,-110,-110,-114,0.51,0.51,0.52,20.16
43838,Nickeil Alexander-Walker,Jaime Jaquez Jr.,Aaron Holiday,20.5,13.5,9.5,over,over,under,25.30,22.29,4.21,4.80,8.79,5.29,-118,-114,-110,0.53,0.52,0.51,18.88
48563,Kyshawn George,Jalen Duren,Jaime Jaquez Jr.,15.5,18.5,13.5,under,under,over,10.77,13.90,22.29,4.73,4.60,8.79,-118,100,-114,0.53,0.49,0.52,18.12
